# 01 — Exploration

Rohdaten laden, verstehen, bereinigen und als Parquet speichern.

**Pipeline:** `Marc21Parser → Cleaner → df_clean.parquet`

**Output:** `data/processed/df_clean.parquet`

In [ ]:
from core.marc21_parser_full import Marc21Parser
from core.cleaning import Cleaner
from core.data_explorer import Marc21Explorer, DDC_MAIN

import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
#import plotly.graph_objects as go

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_RAW       = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

## 1 — Daten einlesen

In [ ]:
FILE_NAME = "dnb-all_hochschulschriften_dnbmarc.mrc.xml.gz"
FILE_PATH = DATA_RAW / FILE_NAME

LIMIT   = None   # None = gesamte Datei
VERBOSE = True

df_raw = Marc21Parser.parse_dnb_theses(str(FILE_PATH), limit=LIMIT, verbose=VERBOSE)
print(f"\nRecords geladen: {len(df_raw):,}")

In [ ]:
print(f"Records: {len(df_raw):,}")
print(f"Spalten: {list(df_raw.columns)}")
print(type(df_raw["084_list"].iloc[0]))  # -> list

## 2 — Rohdaten-Überblick

In [ ]:
explorer_raw = Marc21Explorer(df_raw)
explorer_raw.overview()

In [ ]:
print("=== Memory (df_raw) ===")
print(Cleaner.memory_report(df_raw).to_string())

## 3 — Klassifikationsfelder analysieren

### 3.1 Verteilung der Anzahl Notationen pro Datensatz

In [ ]:
NOTATION_FIELDS = ["082_list", "083_list", "084_list"]

for field in NOTATION_FIELDS:
    stats = explorer_raw.field_stats(field)
    print(f"\n{field}:")
    for k, v in stats.items():
        print(f"  {k}: {v}")

In [ ]:
# Anteil leerer Notationsfelder
no_082  = df_raw["082_list"].apply(len) == 0
no_083  = df_raw["083_list"].apply(len) == 0
no_084  = df_raw["084_list"].apply(len) == 0
no_any  = no_082 & no_083 & no_084

print("=== Klassifikationsabdeckung (df_raw) ===")
print(f"ohne 082:              {no_082.sum():>8,} ({no_082.mean()*100:.1f}%)")
print(f"ohne 083:              {no_083.sum():>8,} ({no_083.mean()*100:.1f}%)")
print(f"ohne 084:              {no_084.sum():>8,} ({no_084.mean()*100:.1f}%)")
print(f"komplett unklassif.:  {no_any.sum():>8,} ({no_any.mean()*100:.1f}%)")

### 3.2 SNDB in Marc21-Feld 083

In [ ]:
def eda_083(df: pd.DataFrame) -> pd.DataFrame:
    """
    Übersicht der Klassifikationssysteme in 083$2.

    Zählt Häufigkeiten von Systemcodes (sdnb, ddc, rvk, ...),
    um Datenqualität und Systemverteilung zu analysieren.
    """

    if "083_list" not in df.columns:
        raise ValueError("Spalte '083_list' nicht vorhanden")

    systems = []

    for entries in df["083_list"]:
        if not isinstance(entries, list):
            continue
        for entry in entries:
            if isinstance(entry, dict):
                val = entry.get("2")
                if not val:
                    continue
                if isinstance(val, list):
                    systems.extend(str(v) for v in val if str(v))
                else:
                    systems.append(str(val))

    counts = pd.Series(systems).value_counts()
    pct = (counts / len(df) * 100).round(2)

    return pd.DataFrame({"count": counts, "pct_%": pct})

In [ ]:
print("=== Notationssysteme in 083$2 ===")
print(eda_083(df_raw).head(20).to_string())

### 3.3 DDC — wie häufig stimmt der DDC-Eintrag in Marc21-Feld 084 mit 082 überein?

In [ ]:
def check_ddc_in_084(df: pd.DataFrame) -> None:
    """Prüft ob und wie konsistent DDC-Codes in 084 ($2=ddc) mit 082 übereinstimmen."""

    def has_ddc_in_084(entries):
        return isinstance(entries, list) and any(
            isinstance(e, dict) and "ddc" in str(e.get("2", "")).lower()
            for e in entries
        )

    mask = df["084_list"].apply(has_ddc_in_084)
    print(f"Records mit DDC in 084: {mask.sum():,} ({mask.mean()*100:.2f}%)")

    def get_codes(lst, filter_ddc=False):
        codes = []
        for e in lst:
            if not isinstance(e, dict):
                continue
            if filter_ddc and "ddc" not in str(e.get("2", "")).lower():
                continue
            val = e.get("a", [])
            codes.extend(val if isinstance(val, list) else [str(val)])
        return [c.strip()[:3] for c in codes if c.strip()]

    df_both = df[
        mask & (df["082_list"].apply(lambda x: isinstance(x, list) and len(x) > 0))
    ].copy()
    df_both["_082"] = df_both["082_list"].apply(get_codes)
    df_both["_084"] = df_both["084_list"].apply(lambda x: get_codes(x, filter_ddc=True))
    df_both["_match"] = df_both.apply(
        lambda r: bool(set(r["_082"]) & set(r["_084"])), axis=1
    )

    print(f"Auch in 082 vorhanden:      {len(df_both):,}")
    print(f"Übereinstimmung:            {df_both['_match'].sum():,} ({df_both['_match'].mean()*100:.1f}%)")
    print("\nFazit: 084-DDC ist Fremddaten → wird ignoriert")


check_ddc_in_084(df_raw)

## 4 — Publikationsjahr

In [ ]:
year_raw = pd.to_numeric(df_raw["publication_year"], errors="coerce")
year_valid = year_raw[year_raw.between(1800, 2030)]

print(f"Gesamt Records:         {len(df_raw):,}")
print(f"Gültige Jahresangaben:  {len(year_valid):,} ({len(year_valid)/len(df_raw)*100:.1f}%)")
print(f"Ungültig / fehlend:     {len(df_raw)-len(year_valid):,}")
print(f"\nSpanne: {int(year_valid.min())} – {int(year_valid.max())}")

# Datenmüll anzeigen
invalid = year_raw[~year_raw.between(1800, 2030)].dropna()
if len(invalid):
    print(f"\nBeispiele ungültiger Jahre: {sorted(invalid.unique())[:10]}")

## 5 — Cleaner anwenden

In [ ]:
df_clean = Cleaner.clean_library_df(df_raw)

print(f"Shape: {df_clean.shape}")
print(f"Spalten: {list(df_clean.columns)}")

In [ ]:
print("=== Memory (df_clean) ===")
print(Cleaner.memory_report(df_clean).to_string())

In [ ]:
explorer_clean = Marc21Explorer(df_clean)
explorer_clean.missing_report()

## 6 — Klassifikationsabdeckung nach Jahr

In [ ]:
# Abdeckungs-Check nach Cleaner
has_082  = df_clean["082_a"].apply(len) > 0
has_083  = df_clean["083_a"].apply(len) > 0
has_sdnb = df_clean["sdnb_codes"].apply(len) > 0
has_any  = has_082 | has_083 | has_sdnb

print("=== Klassifikationsabdeckung (df_clean) ===")
print(f"mit 082 (DDC):     {has_082.sum():>8,} ({has_082.mean()*100:.1f}%)")
print(f"mit 083 (DDC):     {has_083.sum():>8,} ({has_083.mean()*100:.1f}%)")
print(f"mit SDNB:          {has_sdnb.sum():>8,} ({has_sdnb.mean()*100:.1f}%)")
print(f"mit any:           {has_any.sum():>8,} ({has_any.mean()*100:.1f}%)")
print(f"komplett ohne:     {(~has_any).sum():>8,} ({(~has_any).mean()*100:.1f}%)")

## 7 — DDC-Hierarchie

> Benötigt `ddc_primary_3digit` → classification_transform.py

In [ ]:
df_transformed = pd.read_parquet(DATA_PROCESSED / "df_transformed.parquet")
explorer_t = Marc21Explorer(df_transformed)

In [ ]:
df = df_transformed.copy()

df["ddc3"] = df["ddc_primary_3digit"].astype(str).str.strip()
df = df[df["ddc3"].str.match(r"^\d{3}$")]

df["DDC_1_key"] = df["ddc3"].str[0]
df["DDC_1_label"] = df["DDC_1_key"].map(DDC_MAIN).fillna("Unbekannt")
df["DDC_1"] = df["DDC_1_label"] + " (" + df["DDC_1_key"] + "00)"
df["DDC_2"] = df["ddc3"].str[:2] + "0"
df["DDC_3"] = df["ddc3"]

hier = df.groupby(["DDC_1", "DDC_2", "DDC_3"]).size().reset_index(name="count")

fig = px.sunburst(
    hier,
    path=["DDC_1", "DDC_2", "DDC_3"],
    values="count",
    title="DDC-Hierarchie mit Hauptklassen",
)

fig.update_layout(
    height=800,
    margin=dict(l=10, r=10, t=40, b=10)
)

fig.show()

# https://deweysearchde.pansoft.de/webdeweysearch/mainClasses.html?reset=schedules

In [ ]:
# Sunburst 1: Gesamtkorpus (2 Ebenen — bei 3 unlesbar)
fig = explorer_t.plot_sunburst(max_depth=2)
fig.show()

In [ ]:
# Welche DDC-Codes haben die frühen Records wirklich?
df_early = df_transformed[
    df_transformed["publication_year"].between(1920, 1950) &
    df_transformed["ddc_primary_3digit"].ne("")
]
print(f"Records 1920–1950 mit DDC: {len(df_early):,}")
print(df_early["ddc_primary_3digit"].value_counts().head(10))
print(df_early["title"].sample(10, random_state=42).to_string())

#300er Sozialwissenschaften 600 Technik, aber 610 Medizin

## 8 – Mineralogie und Lagerstättenkunde im gesamten Datenset

In [ ]:
#Frequenzanalyse für Sachgruppen Geowissenschaften (31) und Bergbau etc. (38)

def count_sdnb_codes_np(df, codes=("31","38")):
    hits = {c: 0 for c in codes}
    total = 0

    for arr in df["sdnb_codes"]:
        if arr is None:
            continue

        # numpy array → in Python-Liste wandeln
        try:
            lst = arr.tolist()
        except AttributeError:
            lst = arr

        if not isinstance(lst, list):
            continue

        total += len(lst)
        for c in lst:
            if str(c).strip() in hits:
                hits[str(c).strip()] += 1

    return pd.DataFrame({"code": list(hits.keys()), "count": list(hits.values())}), total

df_codes, total = count_sdnb_codes_np(df_clean)
print(total)
print(df_codes)

In [ ]:
# Titelsuche: Begriffe im Gesamtkorpus
TERMS_GEO = {
    "lagerstätte": "Lagerstätte",
    "geochemie":   "Geochemie",
    "mineralogie": "Mineralogie",
    "petrologie":  "Petrologie",
    "petrographie":"Petrographie",
    "sediment":    "Sediment",
    "geologie":    "Geologie",
    "kristall":    "Kristall",
    "hydrogeologie":"Hydrogeologie",
    "erz":"Erz",
}

def count_term(term, df):
    mask = df["title"].str.lower().str.contains(term, na=False)
    total      = mask.sum()
    mit_ddc    = (mask & df["ddc_primary_3digit"].ne("")).sum()
    mit_sdnb   = (mask & df["has_sdnb"]).sum()
    unklassif  = (mask & df["ddc_primary_3digit"].eq("") & ~df["has_sdnb"]).sum()
    return {
        "Begriff":        TERMS_GEO[term],
        "Gesamt":         total,
        "Mit DDC":        mit_ddc,
        "Mit SDNB":       mit_sdnb,
        "Unklassifiziert":unklassif,
    }

rows = [count_term(t, df_transformed) for t in TERMS_GEO]
df_terms = pd.DataFrame(rows).sort_values("Gesamt", ascending=False)
print(df_terms.to_string(index=False))

In [ ]:
df_plot = df_terms.melt(
    id_vars="Begriff",
    value_vars=["Mit DDC", "Mit SDNB", "Unklassifiziert"],
    var_name="Status", value_name="Anzahl"
)

fig = px.bar(
    df_plot,
    x="Anzahl", y="Begriff",
    color="Status",
    orientation="h",
    barmode="stack",
    color_discrete_map={
        "Mit DDC":         "#1E88E5",
        "Mit SDNB":        "#FF9800",
        "Unklassifiziert": "#CFD8DC",
    },
    title="Geowiss. Titelbegriffe im DNB-Korpus — klassifiziert vs. unklassifiziert",
    labels={"Anzahl": "Anzahl Records", "Begriff": ""},
    category_orders={"Status": ["Unklassifiziert", "Mit SDNB", "Mit DDC"]},
)
fig.update_layout(
    plot_bgcolor="white", height=450,
    legend=dict(orientation="h", y=1.08, x=0.5, xanchor="center"),
)
fig.update_xaxes(showgrid=True, gridcolor="#EEEEEE")
fig.show()

## 9 — df_clean speichern

In [ ]:
out_path = DATA_PROCESSED / "df_clean.parquet"
df_clean.to_parquet(out_path, engine="pyarrow", compression="snappy")
print(f"Gespeichert: {out_path}")
print(f"Größe:       {out_path.stat().st_size / 1024**2:.1f} MB")